In [4]:
import pandas as pd
import numpy as np


# 1. 데이터 불러오기
df = pd.read_csv("final_softcone.csv")

# 혹시 이전 실행으로 남아있을 수 있는 코멧 관련 컬럼 제거
drop_cols = ["코멧여부", "코멧score", "코멧유입경로"]
df = df.drop(columns=[col for col in drop_cols if col in df.columns])

# 기존 컬럼 저장
original_cols = df.columns.tolist()



# 2. 숫자형 컬럼 안전 처리
num_cols = [
    "X_팔로워",
    "유튜브_구독자",
    "최고_팔로워",
    "유튜브_유입지수"
]

for col in num_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce").fillna(0)



# 3. X 유입지수 생성
#    유튜브_유입지수는 기존 컬럼 사용
df["X_유입지수"] = (
    np.log10(df["X_팔로워"] + 1) /
    np.log10(df["최고_팔로워"] + 1)
)

df["X_유입지수"] = (
    df["X_유입지수"]
    .replace([np.inf, -np.inf], np.nan)
    .fillna(0)
)



# 4. 코멧 후보 조건 생성
# 조건 1: 외부 플랫폼 체급 상위권
top_x = df[df["X_팔로워"] > 0].nlargest(200, "X_팔로워").index
top_yt = df[df["유튜브_구독자"] > 0].nlargest(200, "유튜브_구독자").index

cond1_idx = top_x.union(top_yt)
cond1_mask = df.index.isin(cond1_idx)


# 조건 2: 방송 체급 대비 외부 유입지수 상위권
top_x_inflow = df.nlargest(200, "X_유입지수").index
top_yt_inflow = df.nlargest(200, "유튜브_유입지수").index

cond2_idx = top_x_inflow.union(top_yt_inflow)
cond2_mask = df.index.isin(cond2_idx)



# 5. 코멧여부 생성
df["코멧여부"] = (cond1_mask & cond2_mask).astype(int)

# 제외 대상 처리
df.loc[df["스트리머명"] == "브이레코드", "코멧여부"] = 0



# 6. 코멧유입경로 생성
yt_inflow_cut = df.nlargest(200, "유튜브_유입지수")["유튜브_유입지수"].min()
x_inflow_cut = df.nlargest(200, "X_유입지수")["X_유입지수"].min()

def assign_comet_route(row):
    is_yt = row["유튜브_유입지수"] >= yt_inflow_cut
    is_x = row["X_유입지수"] >= x_inflow_cut

    if is_yt and is_x:
        return "하이브리드"
    elif is_yt:
        return "유튜브 강세형"
    elif is_x:
        return "X 강세형"
    else:
        return "기타"

df["코멧유입경로"] = df.apply(assign_comet_route, axis=1)

# 코멧이 아닌 사람은 비코멧 처리
df.loc[df["코멧여부"] == 0, "코멧유입경로"] = "비코멧"



# 7. 코멧score 생성
#    로그 보정 외부활성도비율 기반

# 의미:
# 방송 체급 대비 외부 팬덤 규모
#
# 기존 단순식:
# (X_팔로워 + 유튜브_구독자) / 최고_팔로워
#
# 변경식:
# log1p(X_팔로워 + 유튜브_구독자) / log1p(최고_팔로워)
#
# 이유:
# 최고_팔로워가 너무 작을 때 점수가 과도하게 튀는 문제 완화

df["_코멧_raw_score"] = (
    np.log1p(df["X_팔로워"] + df["유튜브_구독자"]) /
    np.log1p(df["최고_팔로워"].replace(0, np.nan))
)

df["_코멧_raw_score"] = (
    df["_코멧_raw_score"]
    .replace([np.inf, -np.inf], np.nan)
    .fillna(0)
    .astype("float64")
)

# 코멧 후보 내부에서만 0~100 MinMax
comet_mask = df["코멧여부"] == 1


df["코멧score"] = pd.Series(0.0, index=df.index, dtype="float64")

if comet_mask.sum() > 0:
    comet_raw = df.loc[comet_mask, "_코멧_raw_score"].astype("float64")

    min_val = comet_raw.min()
    max_val = comet_raw.max()

    if max_val == min_val:
        df.loc[comet_mask, "코멧score"] = 100.0
    else:
        comet_score = (
            (comet_raw - min_val)
            / (max_val - min_val)
            * 100
        ).astype("float64")

        df.loc[comet_mask, "코멧score"] = comet_score.values

df["코멧score"] = df["코멧score"].astype("float64").round(2)



# 8. 최종 컬럼 정리
#    기존 컬럼 + 코멧여부 + 코멧score + 코멧유입경로
final_cols = original_cols + ["코멧여부", "코멧score", "코멧유입경로"]
df = df[final_cols]



# 9. 결과 확인
print(f"전체 인원: {len(df)}명")
print(f"코멧 인원: {df['코멧여부'].sum()}명")

print("\n[코멧유입경로 분포]")
print(df[df["코멧여부"] == 1]["코멧유입경로"].value_counts())

print("\n[코멧score 상위 20명]")
print(
    df[df["코멧여부"] == 1]
    .sort_values("코멧score", ascending=False)[
        ["스트리머명", "코멧여부", "코멧score", "코멧유입경로"]
    ]
    .head(20)
)

print("\n[코멧score 100점 인원]")
print((df["코멧score"] == 100).sum())


# 10. 저장
#df.to_csv(
#    "final_softcone_with_comet.csv",
#    index=False,
#    encoding="utf-8-sig"
#)

전체 인원: 11157명
코멧 인원: 68명

[코멧유입경로 분포]
코멧유입경로
X 강세형      50
유튜브 강세형    14
하이브리드       4
Name: count, dtype: int64

[코멧score 상위 20명]
               스트리머명  코멧여부  코멧score   코멧유입경로
10712            앤무디     1   100.00    X 강세형
10895          xesii     1    81.71    하이브리드
8073              냥찬     1    52.87    X 강세형
9999            리아 뇌     1    49.65    하이브리드
3608            나는코히     1    42.98  유튜브 강세형
7161            총며칠ㅡ     1    41.24  유튜브 강세형
8961              수콩     1    35.03    X 강세형
8690            치로미2     1    33.85    X 강세형
5715   DIO fevercell     1    33.20    X 강세형
2403             태비군     1    33.12  유튜브 강세형
6585             리코링     1    33.06    X 강세형
7037             이천일     1    32.96  유튜브 강세형
8039         츠키요미 미호     1    30.41    X 강세형
7055              시신     1    26.61    X 강세형
6464            비하와와     1    25.20    X 강세형
8954          나는 범이요     1    25.10  유튜브 강세형
8855             워니이     1    25.03  유튜브 강세형
4581              우현     1    24.72  유튜브 강세형
7940          

### 코멧의 의미

- 방송판에서는 신입/중소 느낌

- 근데 외부 플랫폼에는 팬이 있음

- 그래서 영입하거나 주목하면 빠르게 성장할 가능성이 있음

### 코멧여부는 어떻게 정해지나?

- 조건1 : 외부 팬덤 절대 규모가 커야 함

    - X_팔로워 TOP 200 또는 유튜브_구독자 TOP 200

- 조건2 : 방송 체급 대비 외부 유입지수가 높아야 함

    - X_유입지수 TOP 200 또는 유튜브_유입지수 TOP 200


### 그래서 코멧이 뭐냐?

1. X 팔로워나 유튜브 구독자가 어느 정도 있음
2. 그런데 최고_팔로워, 즉 방송 플랫폼 체급은 상대적으로 작음
3. 그래서 외부 팬덤이 방송으로 넘어오면 성장 여지가 큼
4. X 강세형 / 유튜브 강세형 / 하이브리드로 나눌 수 있음


### 코멧score란?

- 코멧_raw_score =
log1p(X_팔로워 + 유튜브_구독자) / log1p(최고_팔로워)

- 코멧score는 코멧 후보들 중에서 방송 체급 대비 외부 팬덤 규모가 얼마나 큰지를 나타내는 점수

- 로그 씌운 이유 : 최고_팔로워가 너무 작으면 점수가 말도 안 되게 튈 수 있음

### 점수가 높게 나오는 사람은?

- X 팔로워 + 유튜브 구독자는 큰데 최고_팔로워는 상대적으로 작은 사람

### 코멧유입경로란?

- X_유입지수는 높고
유튜브_유입지수는 상대적으로 덜 높은 유형

- 유튜브_유입지수는 높고
X_유입지수는 상대적으로 덜 높은 유형

- X_유입지수와 유튜브_유입지수가 둘 다 높은 유형

- 코멧여부가 0인 사람 